# TP=1 vs TP=2 benchmark and acceptance verdict

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Same model, prompts and sampling for both runs (one process per TP). Reports measured decode tokens/s, prefill tokens/s, latency and GPU memory, and re-checks the `TRITON_ATTN` engine config; **no speed-up is assumed** (the T4s have no P2P). Ends with the overall acceptance verdict.

## 1. Setup — locate the release bundle

In [1]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    """Download to `<dest>.part`, check free disk space and the advertised length, then replace `dest` atomically; the partial file is removed on any failure."""
    dest = pathlib.Path(dest); part = dest.with_name(dest.name + ".part")
    part.unlink(missing_ok=True)
    try:
        total = _range_ok(url)
        if total and shutil.disk_usage(dest.parent).free < total + (64 << 20):
            raise RuntimeError(f"not enough free disk space in {dest.parent} for {total / 1e6:.0f} MB")
        _download_to(url, part, workers, retries)
        if total and part.stat().st_size != total:
            raise RuntimeError(f"incomplete download: {part.stat().st_size} of {total} bytes")
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def _download_to(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Explicit release identity: only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted
# (an older rc is never silently selected). Source 1: the attached Kaggle dataset. Source 2 (nothing attached): the Hugging Face CDN.
RELEASE = "stokastisk-0.1.0-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading it from Hugging Face (needs Internet ON) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE)

release: stokastisk-0.1.0-kaggle-t4-sm75 | bundle: /kaggle/input/datasets/waqasm86/stokastisk-0-1-0-kaggle-t4-sm75-kaggle-input-2/stokastisk-0.1.0-kaggle-t4-sm75


## 2. Install into this fresh Kaggle session (idempotent, `--no-deps`, SHA256-verified, protected packages untouched)

Kaggle gives every notebook a new session: what notebook 02 installed does not exist here. Internet must be ON.

In [2]:
import subprocess, sys, os
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)],
                   env=dict(os.environ, STOKASTISK_OUT="/kaggle/working/stokastisk-acceptance/prereq-install"))
assert r.returncode == 0, "prerequisite install failed (see output above); do not continue"

PASS SHA256SUMS verified before any install -- all files match
PASS python 3.13 -- 3.13.16
PASS baseline protected (torch/triton exact, transformers>=5.5.3, no prohibited pkgs) -- {'torch': '2.11.0+cu130', 'triton': '3.6.0', 'transformers': '5.20.2', 'numpy': '1.26.4', 'nvidia-cuda-nvcc': '13.0.88', 'nvidia-nvvm': '13.0.88'}
PASS two Tesla T4 (SM75) -- 2 gpu(s) [(7, 5), (7, 5)]
PASS exactly two wheels -- ['stokastisk-0.1.0-py3-none-any.whl', 'vllm-0.25.1-cp38-abi3-linux_x86_64.whl']
PASS wheels contain no FlashAttention/FlashInfer/FlashMLA files -- none
PASS dependency plan has no conflicts (nothing is installed if this fails) -- {'install': {'apache-tvm-ffi': '0.1.9', 'xgrammar': '0.2.5.post1', 'llguidance': '1.7.6', 'cbor2': '6.2.0.post2', 'msgspec': '0.22.0', 'openai-harmony': '0.0.8', 'partial-json-parser': '0.2.1.1.post7', 'pybase64': '1.5.1'}}
PASS download exact pins (no resolver, --no-deps) -- 8 pins
PASS install wheels --force-reinstall --no-deps (replaces any other stokastisk

## 3. Provenance — every executed script and wheel must match `manifests/SHA256SUMS`

In [3]:
import hashlib, glob
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
print("bundle root     :", BUNDLE)
print("scripts/kaggle  :", SCRIPTS)
for s in sorted(SCRIPTS.glob("*.py")): print(f"{sha(s)[:16]}  {s.name}")
for w in sorted((BUNDLE / "wheels").glob("*.whl")): print(f"{sha(w)}  {w.name}")
expected = {l.split(None, 1)[1].strip(): l.split()[0] for l in (BUNDLE / "manifests" / "SHA256SUMS").read_text().splitlines()}
for rel in [f"scripts/kaggle/{s.name}" for s in SCRIPTS.glob("*.py")] + [f"wheels/{w.name}" for w in (BUNDLE / "wheels").glob("*.whl")]:
    assert expected.get(rel) == sha(BUNDLE / rel), f"{rel} does not match manifests/SHA256SUMS (stale or modified file)"
print("all executed scripts and wheels match manifests/SHA256SUMS")

bundle root     : /kaggle/input/datasets/waqasm86/stokastisk-0-1-0-kaggle-t4-sm75-kaggle-input-2/stokastisk-0.1.0-kaggle-t4-sm75
scripts/kaggle  : /kaggle/input/datasets/waqasm86/stokastisk-0-1-0-kaggle-t4-sm75-kaggle-input-2/stokastisk-0.1.0-kaggle-t4-sm75/scripts/kaggle
efadf12cbcf626f2  01_env_audit.py
676775e7dee8eb0a  02_install.py
fca537b87762b725  03_single_gpu.py
d1d86f0ef59897b2  04_nccl_dual_gpu.py
f72d35ce18f9494f  04_nccl_launch.py
cde39b21e01dc6f7  04_nccl_summary.py
c4c3170114e0226b  05_backend_evidence.py
3056f159b9950b1e  05_inference.py
8f963ee80ba7f842  06_tp_benchmark.py
8cd4eb9644f20fea  07_summary.py
6dec97e2c36eca32  _common.py
3ae5e23a6e98bc85  dual_07_topology.py
ecd1f6cc6133ff25  dual_08_pp_validation.py
8122738f63206932  dual_09_replica_validation.py
145805509e088cbc  dual_10_comm.py
60058bf01a8dc8a4  dual_10_nccl_worker.py
c2b815ecff582700  dual_11_balance.py
993c58154edf9509  dual_12_acceptance.py
dca3a74207e84ae7  dual_13_api_compat.py
e95abbb71827bb19  dua

## 4. The installed packages must be byte-identical to the bundle's wheels

In [4]:
import importlib, importlib.metadata as md, zipfile, hashlib
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]
    mod = importlib.import_module(pkg)
    root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)} imported from {root}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}; uninstall it and re-run the install cell"

stokastisk 0.1.0 imported from /usr/local/lib/python3.13/dist-packages/stokastisk: identical to stokastisk-0.1.0-py3-none-any.whl
vllm 0.25.1 imported from /usr/local/lib/python3.13/dist-packages/vllm: identical to vllm-0.25.1-cp38-abi3-linux_x86_64.whl


## 5. Benchmark Qwen2.5-1.5B-Instruct at TP=1 and TP=2

In [5]:
RC = {}
env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""))
for tp in (1, 2):
    RC[f"06_bench_tp{tp}"] = subprocess.run([sys.executable, str(SCRIPTS / "06_tp_benchmark.py"), str(tp), "Qwen/Qwen2.5-1.5B-Instruct"], cwd=SCRIPTS, env=env).returncode
    print("TP", tp, "rc", RC[f"06_bench_tp{tp}"])

INFO 11-11 15:44:49 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, tq_max_kv_splits_for_cuda_graph=32, use_trtllm_attention=None, disable_flashinfer_q_quantization=False, mla_prefill_backend=None, use_prefill_query_quantization=False, use_fp4_indexer_cache=False, indexer_kv_dtype='bf16', use_non_causal=False, flex_attn_block_m=None, flex_attn_block_n=None, flex_attn_q_block_size=None, flex_attn_kv_block_size=None), 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


INFO 11-11 15:45:04 [model.py:619] Resolved architecture: Qwen2ForCausalLM
WARNING 11-11 15:45:04 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 11-11 15:45:04 [model.py:1776] Using max model len 4096
INFO 11-11 15:45:04 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-11 15:45:04 [vllm.py:1042] Asynchronous scheduling is enabled.
WARNING 11-11 15:45:04 [vllm.py:1096] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 11-11 15:45:04 [vllm.py:1144] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 11-11 15:45:04 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 11-11 15:45:04 [vllm.py:1322] Cudagraph is disabled under eager mode
INFO 11-11 15:

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.90s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.90s/it]
(EngineCore pid=249) 


(EngineCore pid=249) INFO 11-11 15:45:34 [default_loader.py:430] Loading weights took 1.92 seconds
(EngineCore pid=249) INFO 11-11 15:45:35 [model_runner.py:302] Model loading took 2.98 GiB and 24.298452 seconds
(EngineCore pid=249) INFO 11-11 15:45:35 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=249) INFO 11-11 15:45:41 [gpu_worker.py:538] Available KV cache memory: 9.28 GiB
(EngineCore pid=249) INFO 11-11 15:45:41 [kv_cache_utils.py:2146] GPU KV cache size: 347,472 tokens
(EngineCore pid=249) INFO 11-11 15:45:41 [kv_cache_utils.py:2147] Maximum concurrency for 4,096 tokens per request: 84.83x
(EngineCore pid=249) INFO 11-11 15:45:41 [cutedsl_warmup.py:97] Skipping CuTeDSL warmup because no compile units were requested.
(EngineCore pid=249) INFO 11-11 15:46:01 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=249) INFO 11-11 15:46:01 

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=249) WARNING 11-11 15:46:03 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=249) WARNING 11-11 15:46:05 [jit_monitor.py:129] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.


... (8 outputs left out)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

PASS generated expected token count -- 4096
PASS engine config backend == TRITON_ATTN -- TRITON_ATTN


Processed prompts: 100%|██████████| 8/8 [00:00<00:00, 64.03it/s, est. speed input: 77921.62 toks/s, output: 64.60 toks/s]


INFO 11-11 15:48:07 [core_client.py:655] [shutdown] MPClient: start timeout=default
INFO 11-11 15:48:07 [core_client.py:657] [shutdown] MPClient: stopping engine manager
(EngineCore pid=543) INFO 11-11 15:48:07 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=543) INFO 11-11 15:48:07 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=543) INFO 11-11 15:48:07 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=543) INFO 11-11 15:48:07 [core.py:1227] [shutdown] EngineCore: exiting busy loop
(EngineCore pid=543) INFO 11-11 15:48:07 [multiproc_executor.py:426] [shutdown] Executor: waiting for worker exit count=2
(Worker_TP0 pid=549) INFO 11-11 15:48:07 [multiproc_executor.py:790] Parent process exited, terminating worker queues
(EngineCore pid=543) INFO 11-11 15:48:09 [multiproc_executor.py:433] [shutdown] Executor: all workers exited gracefully
INFO 11-11 15:48:09 

## 6. Measured comparison and the overall verdict

In [6]:
import subprocess, sys, os
def run_stage(script, *args):
    """Run a bundled stage script; returns its exit code (a failure is asserted in the final verdict cell, after evidence is packaged)."""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS,
                       env=dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", "")))
    print(f"{script} exit code: {r.returncode}")
    return r.returncode

In [7]:
RC["07_summary"] = run_stage("07_summary.py")

PASS      06_bench_tp1
PASS      06_bench_tp2
NOT_RUN   01_env_audit  (other notebook; informational)
NOT_RUN   02_install  (other notebook; informational)
NOT_RUN   03_single_gpu  (other notebook; informational)
NOT_RUN   04_nccl_dual_gpu  (other notebook; informational)
NOT_RUN   05_inference_tp1  (other notebook; informational)
NOT_RUN   05_inference_tp2  (other notebook; informational)
measured (not assumed): TP1 981.2 tok/s, TP2 528.2 tok/s, ratio 0.54x
NOTEBOOK 06 TP BENCHMARK: PASS
07_summary.py exit code: 0


## Package the evidence

Creates `/kaggle/working/06-stokastisk-acceptance.zip` (download it from the Output tab).

In [8]:
import shutil
print(shutil.make_archive("/kaggle/working/06-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

/kaggle/working/06-stokastisk-acceptance.zip


## Verdict — this cell fails the notebook if any stage failed

In [9]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)

ALL STAGES PASSED {'06_bench_tp1': 0, '06_bench_tp2': 0, '07_summary': 0}
